# 📖 Les 7: Introductie GeoPandas in JupyterLite
In deze les leren we hoe we geografische data kunnen verwerken met **GeoPandas** direct in de browser via **JupyterLite**.

## Deel 1: Packages installeren met `piplite`
In JupyterLite installeren we `pandas`, `shapely` en `matplotlib`. Daarna installeren we `geopandas` zonder C-afhankelijkheden via `deps=False`.

In [ ]:
import piplite

# 1. Pure Python basispakketten installeren
await piplite.install(['pandas', 'shapely', 'matplotlib'])

# 2. GeoPandas installeren zonder C-drivers
await piplite.install('geopandas', deps=False)

import geopandas as gpd
print("GeoPandas versie:", gpd.__version__)

## Deel 2: Handmatig een GeoDataFrame & CRS instellen
Een **GeoDataFrame** is een Pandas DataFrame met een speciale `geometry` kolom. We stellen het Coordinate Reference System (CRS) in op `EPSG:4326` (WGS84 - lengtegraden en breedtegraden).

> **Let op:** In JupyterLite gebruiken we `EPSG:4326` als standaard. Herprojectie via `.to_crs()` werkt niet in de browser omdat de gecompileerde PROJ C-bibliotheek ontbreekt.

In [ ]:
import pandas as pd
from shapely.geometry import Point

# Data met steden en hun coördinaten (Breedtegraad / Lengtegraad)
data = {
    "Stad": ["Amsterdam", "Utrecht", "Rotterdam", "Groningen"],
    "Inwoners": [872000, 360000, 650000, 233000],
    "Lat": [52.3676, 52.0907, 51.9244, 53.2194],
    "Lon": [4.9041, 5.1214, 4.4777, 6.5665]
}

df = pd.DataFrame(data)

# Shapely Points maken (let op de volgorde: Point(longitude, latitude))
geometry = [Point(lon, lat) for lon, lat in zip(df["Lon"], df["Lat"])]

# GeoDataFrame maken met WGS84 coördinatenstelsel (EPSG:4326)
gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")
gdf

## Deel 3: Kaart Visualiseren
Met de `.plot()` methode kun je de ruimtelijke punten op een kaart laten zien.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 8))
gdf.plot(ax=ax, color="red", markersize=gdf["Inwoners"]/10000, alpha=0.7)

# Labels toevoegen per stad
for x, y, label in zip(gdf.geometry.x, gdf.geometry.y, gdf["Stad"]):
    ax.text(x + 0.05, y, label, fontsize=10)

plt.title("Nederlandse Steden (Grootte = Inwoners)")
plt.xlabel("Lengtegraad (Longitude)")
plt.ylabel("Breedtegraad (Latitude)")
plt.grid(True)
plt.show()

## Deel 4: Afstanden berekenen in kilometers (Haversine)
Omdat we geen `.to_crs()` naar een meter-gebaseerd stelsel kunnen gebruiken, berekenen we de afstand over de aarde via de Haversine-formule.

In [ ]:
import math

def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0  # Straal van de aarde in km
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = math.sin(dlat / 2)**2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon / 2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return R * c

# Afstand berekenen vanaf Amsterdam (Lat: 52.3676, Lon: 4.9041)
ams_lat, ams_lon = 52.3676, 4.9041
gdf["Afstand_tot_Amsterdam_km"] = [round(haversine(ams_lat, ams_lon, lat, lon), 1) for lat, lon in zip(gdf["Lat"], gdf["Lon"])]

gdf[["Stad", "Afstand_tot_Amsterdam_km"]]

## Deel 5: Geodata Exporteren & Inlezen
We exporteren en importeren GeoJSON via de standaard Python `json` module.

In [ ]:
import json

# 1. Exporteren naar GeoJSON via .to_json()
geojson_string = gdf.to_json()
with open("steden.geojson", "w") as f:
    f.write(geojson_string)
print("Bestand 'steden.geojson' is opgeslagen!")

# 2. Inlezen vanuit GeoJSON via json.load() en GeoDataFrame.from_features()
with open("steden.geojson", "r") as f:
    geojson_data = json.load(f)

ingelezen_gdf = gpd.GeoDataFrame.from_features(geojson_data["features"], crs="EPSG:4326")
ingelezen_gdf